## Téléchargement des images et collecte des métadonnées

Ce notebook permet de collecter automatiquement des images libres de droit à l'aide d'une requête SPARQL sur Wikidata. Les images sont enregistrées localement dans un dossier dédié, et leurs métadonnées techniques (format, dimensions, orientation, date de prise de vue, modèle d'appareil photo) sont extraites et sauvegardées dans un fichier CSV.

Ce script répond à la première étape du projet : collecte de données.

Les principales opérations réalisées sont :
- Construction et exécution d'une requête SPARQL pour récupérer des images associées à des villes.
- Téléchargement des images (format JPG ou PNG) dans un dossier local.
- Extraction des métadonnées via la bibliothèque Pillow (EXIF).
- Enregistrement des informations dans un fichier CSV pour analyse ultérieure.

Les images utilisées proviennent de Wikidata, et sont donc sous licence libre.

In [2]:
import os
import csv
import requests
from PIL import Image, ExifTags
from io import BytesIO
from tqdm import tqdm

# Configuration
SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"
OUTPUT_CSV = "../metadata/images_metadata.csv"
IMAGE_DIR = "../images"
HEADERS = {"User-Agent": "MassiveDataProject/1.0 (cmvilleroy@gmail.com)"}

# Création des dossiers si nécessaires
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

def build_sparql_query(limit=100):
    return f"""
    SELECT DISTINCT ?ville ?villeLabel ?pays ?paysLabel ?image WHERE {{
      ?ville wdt:P31 wd:Q1549591;
             wdt:P17 ?pays;
             wdt:P18 ?image.
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "fr". }}
    }}
    LIMIT {limit}
    """

def run_sparql_query(query):
    response = requests.get(SPARQL_ENDPOINT, params={"query": query, "format": "json"}, headers=HEADERS)
    response.raise_for_status()
    return response.json()["results"]["bindings"]

def download_image(url, filename):
    try:
        response = requests.get(url, stream=True, timeout=10, headers=HEADERS)
        response.raise_for_status()
        image_path = os.path.join(IMAGE_DIR, filename)
        if not os.path.exists(image_path):
            with open(image_path, "wb") as f:
                for chunk in response.iter_content(1024):
                    f.write(chunk)
        return image_path
    except Exception as e:
        print(f"Erreur lors du téléchargement : {url} | {e}")
        return None

def get_image_metadata(image_path):
    metadata = {
        "format": None,
        "width": None,
        "height": None,
        "orientation": None,
        "capture_date": None,
        "device": None
    }
    try:
        with Image.open(image_path) as img:
            metadata["format"] = img.format
            metadata["width"], metadata["height"] = img.size
            metadata["orientation"] = (
                "portrait" if img.height > img.width
                else "landscape" if img.width > img.height
                else "square"
            )
            exif_data = img._getexif()
            if exif_data:
                exif = {ExifTags.TAGS.get(k, k): v for k, v in exif_data.items()}
                metadata["capture_date"] = exif.get("DateTimeOriginal")
                make = exif.get("Make", "")
                model = exif.get("Model", "")
                metadata["device"] = f"{make} {model}".strip()
    except Exception as e:
        print(f"Erreur lecture métadonnées {image_path} : {e}")
    return metadata

def process_images(limit=100):
    query = build_sparql_query(limit)
    results = run_sparql_query(query)

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as csvfile:
        fieldnames = [
            "ville", "pays", "image_url", "image_filename",
            "format", "width", "height", "orientation",
            "capture_date", "device"
        ]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for entry in tqdm(results, desc="Téléchargement et extraction"):
            ville = entry["villeLabel"]["value"]
            pays = entry["paysLabel"]["value"]
            image_url = entry["image"]["value"]
            image_filename = os.path.basename(image_url).split("?")[0]

            local_path = download_image(image_url, image_filename)
            if local_path:
                meta = get_image_metadata(local_path)
                writer.writerow({
                    "ville": ville,
                    "pays": pays,
                    "image_url": image_url,
                    "image_filename": image_filename,
                    "format": meta["format"],
                    "width": meta["width"],
                    "height": meta["height"],
                    "orientation": meta["orientation"],
                    "capture_date": meta["capture_date"],
                    "device": meta["device"]
                })

# Lancement
process_images(limit=100)


Téléchargement et extraction:  14%|█▍        | 14/100 [00:10<00:57,  1.50it/s]

Erreur lors du téléchargement : http://commons.wikimedia.org/wiki/Special:FilePath/%D0%9A%D1%80%D0%B5%D1%81%D1%82%D0%BE%D0%B2%D0%BE%D0%B7%D0%B4%D0%B2%D0%B8%D0%B6%D0%B5%D0%BD%D1%81%D0%BA%D0%B0%D1%8F%20%D1%86%D0%B5%D1%80%D0%BA%D0%BE%D0%B2%D1%8C%20%28%D0%A2%D1%8E%D0%BC%D0%B5%D0%BD%D1%8C%29-2.jpg | [Errno 2] No such file or directory: '../images\\%D0%9A%D1%80%D0%B5%D1%81%D1%82%D0%BE%D0%B2%D0%BE%D0%B7%D0%B4%D0%B2%D0%B8%D0%B6%D0%B5%D0%BD%D1%81%D0%BA%D0%B0%D1%8F%20%D1%86%D0%B5%D1%80%D0%BA%D0%BE%D0%B2%D1%8C%20%28%D0%A2%D1%8E%D0%BC%D0%B5%D0%BD%D1%8C%29-2.jpg'


Téléchargement et extraction:  22%|██▏       | 22/100 [00:17<01:01,  1.27it/s]

Erreur lors du téléchargement : http://commons.wikimedia.org/wiki/Special:FilePath/%D0%90%D1%80%D1%85%D0%B8%D1%82%D0%B5%D0%BA%D1%82%D1%83%D1%80%D0%BD%D1%8B%D0%B9%20%D0%B0%D0%BD%D1%81%D0%B0%D0%BC%D0%B1%D0%BB%D1%8C%20%D0%A2%D1%80%D0%B8%D1%84%D0%BE%D0%BD%D0%BE%D0%B2%D0%B0%20%D0%BC%D0%BE%D0%BD%D0%B0%D1%81%D1%82%D1%8B%D1%80%D1%8F%2C%20%D0%B2%D0%B8%D0%B4%20%D1%81%20%D0%BE%D0%B7%D0%B5%D1%80%D0%B0%20%D0%BE%D1%81%D0%B5%D0%BD%D1%8C%D1%8E.jpg | [Errno 2] No such file or directory: '../images\\%D0%90%D1%80%D1%85%D0%B8%D1%82%D0%B5%D0%BA%D1%82%D1%83%D1%80%D0%BD%D1%8B%D0%B9%20%D0%B0%D0%BD%D1%81%D0%B0%D0%BC%D0%B1%D0%BB%D1%8C%20%D0%A2%D1%80%D0%B8%D1%84%D0%BE%D0%BD%D0%BE%D0%B2%D0%B0%20%D0%BC%D0%BE%D0%BD%D0%B0%D1%81%D1%82%D1%8B%D1%80%D1%8F%2C%20%D0%B2%D0%B8%D0%B4%20%D1%81%20%D0%BE%D0%B7%D0%B5%D1%80%D0%B0%20%D0%BE%D1%81%D0%B5%D0%BD%D1%8C%D1%8E.jpg'


Téléchargement et extraction:  49%|████▉     | 49/100 [00:38<00:40,  1.25it/s]

...

Téléchargement et extraction:  55%|█████▌    | 55/100 [00:43<00:32,  1.40it/s]

Erreur lors du téléchargement : http://commons.wikimedia.org/wiki/Special:FilePath/%D0%98%D1%80%D0%BA%D1%83%D1%82%D1%81%D0%BA.%20%D0%9D%D0%B8%D0%B6%D0%BD%D1%8F%D1%8F%20%D0%9D%D0%B0%D0%B1%D0%B5%D1%80%D0%B5%D0%B6%D0%BD%D0%B0%D1%8F.JPG | [Errno 2] No such file or directory: '../images\\%D0%98%D1%80%D0%BA%D1%83%D1%82%D1%81%D0%BA.%20%D0%9D%D0%B8%D0%B6%D0%BD%D1%8F%D1%8F%20%D0%9D%D0%B0%D0%B1%D0%B5%D1%80%D0%B5%D0%B6%D0%BD%D0%B0%D1%8F.JPG'


Téléchargement et extraction:  90%|█████████ | 90/100 [01:12<00:07,  1.39it/s]

Erreur lors du téléchargement : http://commons.wikimedia.org/wiki/Special:FilePath/%D0%9F%D1%80%D0%BE%D1%81%D0%BF%D0%B5%D0%BA%D1%82%20%D0%9B%D0%B5%D0%BD%D0%B8%D0%BD%D0%B0%20%D0%B2%D0%BE%D0%B7%D0%BB%D0%B5%20%D0%BE%D1%81%D1%82%D0%B0%D0%BD%D0%BE%D0%B2%D0%BA%D0%B8%20%C2%AB%D0%9A%D0%B8%D0%BD%D0%BE%D1%82%D0%B5%D0%B0%D1%82%D1%80%20%D0%A6%D0%B5%D0%BD%D1%82%D1%80%D0%B0%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9%C2%BB.jpg | [Errno 2] No such file or directory: '../images\\%D0%9F%D1%80%D0%BE%D1%81%D0%BF%D0%B5%D0%BA%D1%82%20%D0%9B%D0%B5%D0%BD%D0%B8%D0%BD%D0%B0%20%D0%B2%D0%BE%D0%B7%D0%BB%D0%B5%20%D0%BE%D1%81%D1%82%D0%B0%D0%BD%D0%BE%D0%B2%D0%BA%D0%B8%20%C2%AB%D0%9A%D0%B8%D0%BD%D0%BE%D1%82%D0%B5%D0%B0%D1%82%D1%80%20%D0%A6%D0%B5%D0%BD%D1%82%D1%80%D0%B0%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9%C2%BB.jpg'


Téléchargement et extraction: 100%|██████████| 100/100 [01:20<00:00,  1.25it/s]
